# Sentence Transformer

- Try BERT instead of SBERT
- Try to find a different SBERT model that's designed for HR
- Replace human resources with hr and try stemming
- Try FastText model
- Look up markov chain monte carlo
- Try alternatives to cosine similarity - BERT has in inbuilt one?

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import PorterStemmer

In [33]:
data = pd.read_csv('../data/potential_talents_cleaned.csv')

In [34]:
data.head()

,id,job_title,location,connection,fit,job_title_stem
0,1,2019 c t bauer college of business graduate ...,"Houston, Texas",85,NaN,2019 c t bauer colleg of busi graduat magna cu...
1,2,native english teacher at epik english progra...,Kanada,500+,NaN,nativ english teacher at epik english program ...
2,3,aspiring hr professional,"Raleigh-Durham, North Carolina Area",44,NaN,aspir hr profession
3,4,people development coordinator at ryan,"Denton, Texas",500+,NaN,peopl develop coordin at ryan
4,5,advisory board member at celal bayar university,"İzmir, Türkiye",500+,NaN,advisori board member at celal bayar univers


In [41]:
# choosing sentence transformer model to use
model = SentenceTransformer('all-MiniLM-L6-v2')

# creating embeddings of the target job title
stemmer = PorterStemmer()
target_title = 'aspiring hr'
target_stem = ' '.join(
    stemmer.stem(word)
    for word in target_title.split()
    )
target_embedding = model.encode([target_stem])

# creating embeddings of the words in the stemmed job titles
job_embeddings = model.encode(data['job_title_stem'].to_list())

# list different types of similarity scorers
similarity_scorers = ['cosine', 'dot', 'euclidean', 'manhattan']

# using each scorer to assess the similarities
for scorer in similarity_scorers:
    model.similarity_fn_name = scorer
    similarities = model.similarity(target_embedding, job_embeddings).flatten()
    data[f'{scorer}_similarity'] = similarities
    data[f'{scorer}_rank'] = (
        data[f'{scorer}_similarity']
        .rank(ascending=False, method='min')
        .astype(int)
    )

# display results
data = data.sort_values(by='cosine_rank', ascending=True).reset_index(drop=True)
data[['job_title', 'job_title_stem', 'cosine_rank', 'dot_rank', 'euclidean_rank', 'manhattan_rank']]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,job_title,job_title_stem,cosine_rank,dot_rank,euclidean_rank,manhattan_rank
0,aspiring hr specialist,aspir hr specialist,1,1,1,1
1,aspiring hr professional,aspir hr profession,2,2,2,2
2,aspiring hr professional,aspir hr profession,2,2,2,2
3,aspiring hr professional an energetic and te...,aspir hr profession an energet and team focus ...,4,4,4,4
4,liberal arts major aspiring hr analyst,liber art major aspir hr analyst,5,5,5,5
5,aspiring hr manager seeking internship in hr,aspir hr manag seek internship in hr,6,6,6,6
6,business management major and aspiring hr manager,busi manag major and aspir hr manag,7,7,7,7
7,seeking hr opportunities,seek hr opportun,8,8,8,9
8,aspiring hr management student seeking an inte...,aspir hr manag student seek an internship,9,9,9,8
9,experienced retail manager and aspiring hr pro...,experienc retail manag and aspir hr profession,10,10,10,10
